Inseption-LSTM Model (CNN + RNN with Handcrafted Features)

Imports

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Flatten, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Definitions

In [2]:
# Load and preprocess images
def load_images(image_paths):
    images = []
    for path in image_paths:
        img = tf.keras.preprocessing.image.load_img(path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = tf.keras.applications.inception_v3.preprocess_input(img)
        images.append(img)
    return np.array(images)

Process

In [3]:
# Load handcrafted features
data = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = data["label"].values
handcrafted_features = data.iloc[:, 2:].values  # Skip filename and label columns

In [4]:
# Load images for CNN
image_size = (299, 299)  # InceptionV3 default input size
image_paths = [f"/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" for file, label in zip(data["filename"], labels)]

In [5]:
# Load images
images = load_images(image_paths)

# Split dataset
X_train_img, X_test_img, X_train_handcrafted, X_test_handcrafted, y_train, y_test = train_test_split(
    images, handcrafted_features, labels, test_size=0.2, random_state=42, stratify=labels)

In [6]:
# CNN Feature Extractor (InceptionV3)
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)

# LSTM for sequence modeling
lstm_input = Input(shape=(X_train_handcrafted.shape[1], 1))  # Handcrafted features
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)

# Combine CNN and LSTM
merged = Concatenate()([cnn_output, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define hybrid model
model = Model(inputs=[inception_base.input, lstm_input], outputs=output_layer)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Train model
model.fit(
    [X_train_img, X_train_handcrafted.reshape(-1, X_train_handcrafted.shape[1], 1)], y_train,
    validation_data=([X_test_img, X_test_handcrafted.reshape(-1, X_test_handcrafted.shape[1], 1)], y_test),
    epochs=25, batch_size=32)

In [ ]:
# Save model
model.save("../Model/signature_forgery_detection_model.h5")

print("Model training completed and saved!")

[ Optional: For Testing Purposes ] Load Model

In [7]:
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model("/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Model/signature_forgery_detection_model.h5")

Evaluation

In [8]:

# Predict on test set
y_pred_probs = model.predict([X_test_img, X_test_handcrafted.reshape(-1, X_test_handcrafted.shape[1], 1)])
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

# Compute evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


98/98 ━━━━━━━━━━━━━━━━━━━━ 98s 991ms/step
Accuracy: 0.9620
Precision: 0.9452
Recall: 0.9808
F1-score: 0.9627
Confusion Matrix:
[[1475   89]
 [  30 1534]]


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Forged', 'Genuine'], yticklabels=['Forged', 'Genuine'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend()
plt.show()

In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred)
plt.plot(recall, precision, color='green', label="Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

In [ ]:
# Distribution of Predictions
sns.histplot(y_pred, bins=20, kde=True)
plt.xlabel("Prediction Confidence")
plt.ylabel("Frequency")
plt.title("Distribution of Model Predictions")
plt.show()

print("Advanced testing completed with visualizations!")